# ITCS 6169/8169 — Assignment 2: results summary from saved checkpoints

Reloads every final model in `checkpoints/` (SigLIP CNN, BYOL CNN, supervised ViT, SSAT ViT; 64px and 128px; every seed found), **re-evaluates each on the validation and test sets**, and aggregates by method and resolution. Nothing is trained here, and no number is copied from an earlier notebook output.

Two checkpoints are stored per model, so two rows of results are reported:
- **best-val checkpoint** — the epoch with the highest validation accuracy. Its *validation* accuracy is optimistically biased (it was selected on validation); its *test* accuracy is an honest estimate because test was never used for selection.
- **last-epoch checkpoint** — no selection at all, so both numbers are unbiased. Use this one if you want a validation number that is not inflated by picking the best epoch.

Run it from the same project folder as the other notebooks (it needs `data/` and `checkpoints/`). It only evaluates, so it takes a few minutes.

In [ ]:
# Core imports
from pathlib import Path
import random, time, copy, math, json, gc, threading, queue

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from PIL import Image

try:
    import torch_directml  # to allow for AMD gpu usage
    HAS_DML = torch_directml.is_available()
except ImportError:
    HAS_DML = False

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"DirectML available: {HAS_DML}")

In [ ]:
PROJECT_ROOT = Path('.')
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

print('Project root:', PROJECT_ROOT.resolve())
print('Training data:', TRAIN_ROOT)
print('Testing data:', TEST_ROOT)

In [ ]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    """CUDA if available, else DirectML (AMD GPU on Windows), else CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if HAS_DML:
        return torch_directml.device()
    return torch.device("cpu")

set_random_seed(SEED)
device = get_device()
print('Device:', device)

## Config

In [ ]:
RESOLUTIONS = (64, 128)
MAX_SEEDS = 5                  # looks for seeds 0..MAX_SEEDS-1 and uses whichever checkpoints exist
VAL_FRACTION = 0.20            # must match training (the split is reproduced from SEED)
EVAL_BS = 100
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
CKPT_DIR = PROJECT_ROOT / 'checkpoints'

# display name -> (checkpoint label used at training time, which file stage, model family)
METHODS = [
    ('SigLIP CNN',     'SigLIP CNN', 'finetune', 'cnn'),
    ('BYOL CNN',       'BYOL CNN',   'finetune', 'cnn'),
    ('ViT supervised', 'ViT sup',    'train',    'vit'),
    ('ViT SSAT',       'ViT SSAT',   'train',    'vit'),
]

def slug(name):
    """Same file-name rule the training notebooks used."""
    return ''.join(c.lower() if c.isalnum() else '_' for c in name).strip('_')

def res_prefix(res):
    return '' if res == 64 else f'{res}px_'

## Data
The validation split is **reproduced** from the training notebooks' seeded `random_split` (only indices are needed, so it is identical at every resolution), then each image is resized exactly as in training. The check at the end of the evaluation compares the recomputed validation accuracy against the one logged during training; a mismatch would mean the split or preprocessing differs.

In [ ]:
from PIL import Image

TRAIN_ROOT, TEST_ROOT = PROJECT_ROOT / 'data' / 'train', PROJECT_ROOT / 'data' / 'test'
base_train = datasets.ImageFolder(TRAIN_ROOT)
class_names, num_classes = base_train.classes, len(base_train.classes)
n_total = len(base_train)
val_size = int(round(n_total * VAL_FRACTION))
gen = torch.Generator().manual_seed(SEED)
_, val_subset = random_split(range(n_total), [n_total - val_size, val_size], generator=gen)
val_idx = list(val_subset.indices)

def make_eval_tensors(res):
    tf = transforms.Compose([transforms.Resize((res, res)), transforms.ToTensor(),
                             transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
    Xv = torch.stack([tf(base_train[i][0].resize((res, res), Image.BILINEAR)) for i in val_idx])   # as in training: resize, then eval transform
    yv = torch.tensor([base_train[i][1] for i in val_idx])
    test_ds = datasets.ImageFolder(TEST_ROOT, transform=tf)
    Xt = torch.stack([test_ds[i][0] for i in range(len(test_ds))])
    yt = torch.tensor([test_ds[i][1] for i in range(len(test_ds))])
    return Xv, yv, Xt, yt

data = {res: make_eval_tensors(res) for res in RESOLUTIONS}
for res, (Xv, yv, Xt, yt) in data.items():
    print(f'{res}px: validation {tuple(Xv.shape)} | test {tuple(Xt.shape)}')
print(f'{num_classes} classes | validation images: {len(val_idx)} | test images: {len(data[RESOLUTIONS[0]][3])}')

## Model definitions (needed only to rebuild the saved networks)

In [ ]:
def conv_bn_relu(in_ch, out_ch):
    return nn.Sequential(nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

class TNet(nn.Module):
    def __init__(self, num_classes=16, widths=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        c1, c2, c3, c4 = widths
        self.features = nn.Sequential(
            conv_bn_relu(3, c1), conv_bn_relu(c1, c1), nn.MaxPool2d(2),
            conv_bn_relu(c1, c2), conv_bn_relu(c2, c2), nn.MaxPool2d(2),
            conv_bn_relu(c2, c3), conv_bn_relu(c3, c3), nn.MaxPool2d(2),
            conv_bn_relu(c3, c4), conv_bn_relu(c4, c4))
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c4, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
IMG_SIZE = 64   # only the default of ViTSSAT(img=...); every model below is built with its own resolution

class Attention(nn.Module):
    def __init__(self, dim, heads, drop=0.0):
        super().__init__()
        self.heads, self.scale = heads, (dim // heads) ** -0.5
        self.qkv, self.proj, self.drop = nn.Linear(dim, dim * 3), nn.Linear(dim, dim), nn.Dropout(drop)

    def forward(self, x):
        B, N, C = x.shape
        q, k, v = [t.reshape(B, N, self.heads, C // self.heads).transpose(1, 2) for t in self.qkv(x).chunk(3, dim=-1)]
        attn = ((q @ k.transpose(-2, -1)) * self.scale).softmax(dim=-1)
        return self.drop(self.proj((attn @ v).transpose(1, 2).reshape(B, N, C)))

class Block(nn.Module):
    def __init__(self, dim, heads, drop=0.0, mlp_ratio=4):
        super().__init__()
        self.n1, self.attn, self.n2 = nn.LayerNorm(dim), Attention(dim, heads, drop), nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * mlp_ratio), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(dim * mlp_ratio, dim), nn.Dropout(drop))

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.mlp(self.n2(x))

class ViTSSAT(nn.Module):
    def __init__(self, img=IMG_SIZE, patch=8, dim=192, depth=6, heads=3, dec_dim=128, dec_depth=2, dec_heads=4,
                 drop=0.1, num_classes=None):
        super().__init__()
        num_classes = num_classes or globals()['num_classes']
        self.N, self.patch = (img // patch) ** 2, patch
        # shared encoder
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.N + 1, dim))
        self.blocks = nn.ModuleList([Block(dim, heads, drop) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)                      # primary stream
        # auxiliary stream (training only)
        self.dec_embed = nn.Linear(dim, dec_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dec_dim))
        self.dec_pos = nn.Parameter(torch.zeros(1, self.N, dec_dim))
        self.dec_blocks = nn.ModuleList([Block(dec_dim, dec_heads, 0.0) for _ in range(dec_depth)])
        self.dec_norm = nn.LayerNorm(dec_dim)
        self.dec_pred = nn.Linear(dec_dim, patch * patch * 3)
        for q in (self.cls_token, self.pos_embed, self.dec_pos):
            nn.init.trunc_normal_(q, std=0.02)
        nn.init.normal_(self.mask_token, std=0.02)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            nn.init.zeros_(m.bias)

    def _tokens(self, x):
        return self.patch_embed(x).flatten(2).transpose(1, 2) + self.pos_embed[:, 1:]

    def _encode(self, t):
        z = torch.cat([(self.cls_token + self.pos_embed[:, :1]).expand(t.size(0), -1, -1), t], dim=1)
        for b in self.blocks:
            z = b(z)
        return self.norm(z)

    def forward(self, x):                                            # primary stream: class logits (used by evaluate())
        return self.head(self._encode(self._tokens(x))[:, 0])

    def aux_loss(self, xa, target, sel, unshuf, mask):               # auxiliary stream: reconstruct hidden patches
        vis = self._encode(sel @ self._tokens(xa))[:, 1:]            # encoder sees only the visible patches
        d = self.dec_embed(vis)
        B, K, _ = d.shape
        full = unshuf @ torch.cat([d, self.mask_token.expand(B, self.N - K, -1)], dim=1) + self.dec_pos
        for b in self.dec_blocks:
            full = b(full)
        pred = self.dec_pred(self.dec_norm(full))
        return (((pred - target) ** 2).mean(-1) * mask).sum() / mask.sum()      # MSE on hidden patches only


def vit_kwargs(cfg):
    return {k: cfg[k] for k in ('patch', 'dim', 'depth', 'heads', 'dec_dim', 'dec_depth', 'dec_heads', 'drop')}

## Evaluate every checkpoint
For each (method, resolution, seed) the notebook loads `<label>_final_s<seed>_<stage>.pt`, rebuilds the network, and evaluates **both** stored weight sets. Missing files are listed and skipped.

In [ ]:
@torch.no_grad()
def evaluate_tensors(model, X, y):
    model.eval()
    correct, loss_sum = 0, 0.0
    for i in range(0, len(X), EVAL_BS):
        xb, yb = X[i:i + EVAL_BS].to(device), y[i:i + EVAL_BS].to(device)
        logits = model(xb)
        loss_sum += F.cross_entropy(logits, yb, reduction='sum').item()
        correct += (logits.argmax(1) == yb).sum().item()
    return correct / len(X), loss_sum / len(X)

def build_model(family, ck, res):
    if family == 'cnn':
        return TNet(num_classes=num_classes)
    return ViTSSAT(img=res, num_classes=num_classes, **vit_kwargs(ck['hparams']['cfg']))

rows, missing = [], []
for res in RESOLUTIONS:
    Xv, yv, Xt, yt = data[res]
    for disp, label, stage, family in METHODS:
        for seed in range(MAX_SEEDS):
            name = f'{res_prefix(res)}{label} final s{seed}'
            path = CKPT_DIR / f'{slug(name)}_{stage}.pt'
            if not path.exists():
                if seed < 2:
                    missing.append(path.name)
                continue
            ck = torch.load(path, map_location='cpu', weights_only=False)
            hist = ck['history']['val_acc']
            for which in ('best', 'last'):
                model = build_model(family, ck, res)
                model.load_state_dict(ck[f'{which}_state'])
                model = model.to(device)
                va, vl = evaluate_tensors(model, Xv, yv)
                ta, tl = evaluate_tensors(model, Xt, yt)
                logged = max(hist) if which == 'best' else hist[-1]
                rows.append(dict(method=disp, res=res, seed=seed, state=which, val_acc=va, test_acc=ta,
                                 val_loss=vl, test_loss=tl, logged_val=logged, file=path.name))
                del model
            print(f'{name:<34} best: val {rows[-2]["val_acc"]:.4f} test {rows[-2]["test_acc"]:.4f} | '
                  f'last: val {rows[-1]["val_acc"]:.4f} test {rows[-1]["test_acc"]:.4f}')
if missing:
    print('\nMissing checkpoints (skipped):', missing)
print(f'\n{len(rows)} (model, checkpoint) evaluations')

In [ ]:
# Consistency check: recomputed validation accuracy vs. the value logged during training (a 1-image difference = 0.0021)
bad = [r for r in rows if abs(r['val_acc'] - r['logged_val']) > 0.0045]
print(f'{len(rows) - len(bad)}/{len(rows)} evaluations match the training log within 2 validation images.')
for r in bad:
    print(f"  MISMATCH {r['file']} ({r['state']}): recomputed {r['val_acc']:.4f} vs logged {r['logged_val']:.4f}")
if bad:
    print('A mismatch means the reproduced validation split or the preprocessing differs from training; do not trust those rows.')

## Summary tables
Means are over the seeds found (the `n` column). With only 2 seeds a standard deviation is not meaningful, so the **min–max** of the per-seed values is shown instead. `test SE` is the binomial standard error of a single model's test accuracy on 400 images (about 1.8 pts), which is usually larger than the seed-to-seed spread.

In [ ]:
def agg(method, res, state, key):
    v = [r[key] for r in rows if r['method'] == method and r['res'] == res and r['state'] == state]
    return v

def fmt_range(v):
    return f'{min(v):.3f}-{max(v):.3f}' if len(v) > 1 else '  -  '

summary = []
for state, title in [('best', 'BEST-VAL CHECKPOINT  (val is optimistic - selected on val; test is honest)'),
                     ('last', 'LAST-EPOCH CHECKPOINT  (no selection: val and test both unbiased)')]:
    print(f'\n=== {title} ===')
    print(f"{'method':<16}{'res':>5}{'n':>3}{'mean val':>10}{'val range':>13}{'mean test':>11}{'test range':>13}{'test SE':>9}")
    for res in RESOLUTIONS:
        for disp, *_ in METHODS:
            va, ta = agg(disp, res, state, 'val_acc'), agg(disp, res, state, 'test_acc')
            if not va:
                continue
            mv, mt = float(np.mean(va)), float(np.mean(ta))
            se = math.sqrt(mt * (1 - mt) / len(data[res][3]))
            print(f'{disp:<16}{str(res)+"px":>5}{len(va):>3}{mv:>10.4f}{fmt_range(va):>13}{mt:>11.4f}{fmt_range(ta):>13}{se:>9.4f}')
            summary.append(dict(state=state, method=disp, res=res, n=len(va), mean_val=mv, mean_test=mt,
                                val_min=min(va), val_max=max(va), test_min=min(ta), test_max=max(ta), test_se=se))

In [ ]:
# Compact table for the report: methods x resolution, "mean val / mean test"  (best-val checkpoint)
for state, title in [('best', 'best-val checkpoint'), ('last', 'last-epoch checkpoint')]:
    print(f'\nmean val / mean test  ({title})')
    print(f"{'method':<16}" + ''.join(f'{str(r)+"px":>20}' for r in RESOLUTIONS))
    for disp, *_ in METHODS:
        cells_ = []
        for res in RESOLUTIONS:
            s = [x for x in summary if x['state'] == state and x['method'] == disp and x['res'] == res]
            cells_.append(f"{s[0]['mean_val']:.3f} / {s[0]['mean_test']:.3f} (n={s[0]['n']})" if s else '-')
        print(f'{disp:<16}' + ''.join(f'{c:>20}' for c in cells_))

print('\nChange from 64px to 128px (mean test, best-val checkpoint):')
for disp, *_ in METHODS:
    a = [x for x in summary if x['state'] == 'best' and x['method'] == disp and x['res'] == 64]
    b = [x for x in summary if x['state'] == 'best' and x['method'] == disp and x['res'] == 128]
    if a and b:
        print(f"  {disp:<16}{b[0]['mean_test'] - a[0]['mean_test']:+.4f}   (val {b[0]['mean_val'] - a[0]['mean_val']:+.4f})")

In [ ]:
# Per-seed rows (raw numbers behind every mean above)
print(f"{'method':<16}{'res':>5}{'seed':>5}{'state':>6}{'val':>9}{'test':>9}{'val loss':>10}{'test loss':>11}")
for r in sorted(rows, key=lambda r: (r['res'], [m[0] for m in METHODS].index(r['method']), r['seed'], r['state'])):
    print(f"{r['method']:<16}{str(r['res'])+'px':>5}{r['seed']:>5}{r['state']:>6}{r['val_acc']:>9.4f}{r['test_acc']:>9.4f}{r['val_loss']:>10.4f}{r['test_loss']:>11.4f}")

In [ ]:
# Save everything in one place
import csv
out_csv, out_json = CKPT_DIR / 'results_per_model.csv', CKPT_DIR / 'results_summary.json'
with open(out_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
out_json.write_text(json.dumps(summary, indent=1))
print('Saved:', out_csv, '|', out_json)

In [ ]:
# Chart: mean test accuracy (best-val checkpoint) with per-seed points; dashed line = chance
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
width = 0.38
for ax, key, ttl in zip(axes, ('test_acc', 'val_acc'), ('Test accuracy', 'Validation accuracy (best-val checkpoint: optimistic)')):
    for j, res in enumerate(RESOLUTIONS):
        means, xs = [], []
        for i, (disp, *_) in enumerate(METHODS):
            v = agg(disp, res, 'best', key)
            if v:
                ax.bar(i + (j - 0.5) * width, np.mean(v), width, color=['tab:blue', 'tab:orange'][j], label=f'{res}px' if i == 0 else None)
                ax.scatter([i + (j - 0.5) * width] * len(v), v, color='k', s=14, zorder=3)
    ax.axhline(1 / num_classes, color='gray', ls='--', lw=1)
    ax.set_xticks(range(len(METHODS))); ax.set_xticklabels([m[0] for m in METHODS], rotation=15)
    ax.set_title(ttl); ax.set_ylim(0, 1)
axes[0].set_ylabel('accuracy'); axes[0].legend()
plt.tight_layout(); plt.show()

## Reading these tables honestly
- **Test accuracy is the number to report** from the best-val checkpoint; the models were selected on validation only, and test was never used for any decision.
- **Validation accuracy of the best-val checkpoint is biased upward** (best of 100 epochs). If you report a validation number, say so, or report the last-epoch row instead.
- **Differences smaller than about 2–3 points are not distinguishable** (400 test images: SE ≈ 1.8 pts per model; 2 seeds).
- The CNN rows include SSL pretraining (200 epochs at 64px, 100 at 128px) plus 100 fine-tuning epochs; the ViT rows are 100 epochs total, so the comparison is not compute-matched.